# Social Networks BO Sweep

Run BO regret sweeps for `GRF+Thompson`, `BFS`, `DFS`, and `RandomSearch`.

In [6]:
%load_ext autoreload
%autoreload 2

import os
import sys
from datetime import datetime
from pathlib import Path

project_root = os.path.abspath('../../..')
sys.path.append(project_root)
sys.path.append(os.path.abspath('.'))

import numpy as np
import pandas as pd
import torch

from experiments.bayesopt.bo_utils import GRFThompson, bfs, build_neighbor_lists, dfs, random_search, run_bo
from grf_gp.sampler import GRFSampler
from data_utils import prepare_social_network_data

CONFIG = {
    'datasets': ['enron', 'facebook', 'twitch', 'youtube'],
    'seeds': [0, 1, 2],
    'algorithms': ['GRF+Thompson', 'BFS', 'DFS', 'RandomSearch'],
    'n_bo_steps': 10,
    'bo_batch_size': 1000,
    'max_walk_length': 3,
    'walks_per_node': 10_000,
    'p_halt': 0.5,
    'train_lr': 0.01,
    'train_iters': 100,
}
CONFIG['gp_retrain_interval'] = 5 * CONFIG['bo_batch_size']

RESULTS_DIR = Path(project_root) / 'experiments' / 'bayesopt' / 'social_networks' / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DATASETS = {}
for dataset_name in CONFIG['datasets']:
    data = prepare_social_network_data(dataset_name, dtype=np.float64)
    y = data['y']
    y_std = y.std()
    if y_std == 0:
        raise ValueError(f'Dataset {dataset_name} has zero target variance.')

    DATASETS[dataset_name] = {
        'dataset_name': dataset_name,
        'X': data['X'],
        'Y': y,
        'Y_norm': (y - y.mean()) / y_std,
        'adjacency': data['adjacency'],
        'neighbors': build_neighbor_lists(data['adjacency'], data['num_nodes']),
        'ground_truth_best': float(y.max()),
        'num_nodes': data['num_nodes'],
        'num_edges': data['num_edges'],
    }


def build_context(dataset_name, seed):
    set_seed(seed)
    base = DATASETS[dataset_name]
    adjacency = base['adjacency']
    adjacency_torch = torch.sparse_csr_tensor(
        torch.tensor(adjacency.indptr, dtype=torch.int64),
        torch.tensor(adjacency.indices, dtype=torch.int64),
        torch.tensor(adjacency.data, dtype=torch.float64),
        size=adjacency.shape,
        device=device,
    )

    sampler = GRFSampler(
        adjacency_torch,
        walks_per_node=CONFIG['walks_per_node'],
        p_halt=CONFIG['p_halt'],
        max_walk_length=CONFIG['max_walk_length'],
        seed=seed,
        use_tqdm=False,
    )

    return {
        **base,
        'rw_mats': sampler(),
    }


def build_algorithms(context):
    return {
        'GRF+Thompson': GRFThompson(
            context=context,
            max_walk_length=CONFIG['max_walk_length'],
            batch_size=CONFIG['bo_batch_size'],
            retrain_interval=CONFIG['gp_retrain_interval'],
            train_lr=CONFIG['train_lr'],
            train_iters=CONFIG['train_iters'],
            device=device,
        ),
        'BFS': bfs,
        'DFS': dfs,
        'RandomSearch': random_search,
    }


[(name, DATASETS[name]['num_nodes'], DATASETS[name]['num_edges']) for name in CONFIG['datasets']]


[('enron', 36692, 183831),
 ('facebook', 22470, 171002),
 ('twitch', 168114, 6797557),
 ('youtube', 1134890, 2987624)]

In [ ]:
all_records = []

for dataset_name in CONFIG['datasets']:
    base = DATASETS[dataset_name]
    print(f"\n=== dataset={dataset_name} ===")
    print(
        f"nodes={base['num_nodes']:,} | edges={base['num_edges']:,} | "
        f"gt_best={base['ground_truth_best']:.1f}"
    )

    for seed in CONFIG['seeds']:
        print(f"\nseed={seed}")
        context = build_context(dataset_name, seed)
        algorithms = build_algorithms(context)

        for algorithm_name in CONFIG['algorithms']:
            print(f"\n--- {algorithm_name} ---")
            result = run_bo(
                name=algorithm_name,
                select_batch=algorithms[algorithm_name],
                context=context,
                n_bo_steps=CONFIG['n_bo_steps'],
                batch_size=CONFIG['bo_batch_size'],
                seed=seed,
                verbose=True,
            )
            for row in result['records']:
                all_records.append(
                    {
                        'dataset': dataset_name,
                        'seed': seed,
                        'algorithm': algorithm_name,
                        'step': row['step'],
                        'visited': row['visited'],
                        'best_value': row['best_value'],
                        'regret': row['regret'],
                        'ground_truth_best': context['ground_truth_best'],
                        'walks_per_node': CONFIG['walks_per_node'],
                        'bo_batch_size': CONFIG['bo_batch_size'],
                        'max_walk_length': CONFIG['max_walk_length'],
                    }
                )

results_df = pd.DataFrame(all_records)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = RESULTS_DIR / f'social_networks_bo_sweep_{timestamp}.csv'
results_df.to_csv(csv_path, index=False)
print(f"\nSaved results to {csv_path}")
results_df.head()